## requires 
# pip install snowflake-connector-python[pandas]
# pip install sqlalchemy


In [2]:
from audioop import add
from pickle import TRUE
import snowflake.connector
import pandas as pd
import datetime
from sqlalchemy import VARCHAR, false, table
import sys
import re
import os

In [3]:
#utilities
def df(conn,sql):
    connCur = conn.cursor()
    connCur.execute(sql)
    df = pd.DataFrame(connCur.fetch_pandas_all())
    connCur.close()
    return df

def getDDL(conn,sql):
    connCur = conn.cursor()
    result = connCur.execute(sql).fetchone()
    connCur.close()
    return result

def getFile(path):
    f = open(path,'r')
    filein = f.readlines()
    f.close()
    return filein

def writeFile(path,fileOut):
    f = open(path,'w')
    f.write(fileOut)
    f.close()

def getFileList(path):
    # get file list for path
    filesList = []
    for x in os.listdir(path):
        z = x.upper()
        if z.endswith('.SQL') and not z.startswith('V_'):
            filesList.append(path + "\\" + x)
    return filesList

In [4]:
#set vars
user = "varun.x.tiwari@us.tel.com"
sourceSchema = "SAPH_USR2"
targetSchema = "SAPH_USR2"
db = "LZ_DEV"
path = "C:\\Source\\Repos\\EnterpriseCloud\\Snow\\K.22547.02_gmxLZTables"
configFile = "C:\\Source\\Repos\\EnterpriseCloud\\Configuration\\FileUploader.config"

In [6]:
## create snowflake connection
conn = snowflake.connector.connect(user=user,
                                    account="telus-global.privatelink", 
                                    client_session_keep_alive = False,
                                    warehouse= "developer_wh", 
                                    database= db,  
                                    role= "sysadmin_dev",
                                    schema= targetSchema,
                                    threads= 1,    
                                    authenticator="externalbrowser",
                                    autocommit=True)  

In [7]:
#generate list of tables from source schema
sql = """select table_name from information_schema.tables 
                            where table_type = 'BASE TABLE'
                            and table_schema = '""" + sourceSchema + """'
                            and table_name not like 'ERROR%';"""
#print(sql)
df_tables = df(conn,sql)
print(df_tables)

                        TABLE_NAME
0                         FILELIST
1       PARTS_EXPORTIMPORT_CONTROL
2       PSL_ZZM_EV_FISCALPERIODEXT
3      SAP_AAM_EVF_ASSETNUMBER_GMX
4       SAP_AAM_EVF_FIXEDASSET_GMX
..                             ...
103      SAP_MMT_EVF_INVENTORY_GMX
104          SAP_SDM_EV_SALESGROUP
105  SAP_SDT_EVF_DELIVERYORDER_GMX
106       SAP_US_MMT_EV_MATDOC_GMX
107             SAP_US_COT_EV_COBK

[108 rows x 1 columns]


In [ ]:
# Generate ddl files for LZ tables for target schema

df_tables['ddl'] = """create or replace TABLE """ + targetSchema + "." + df_tables['TABLE_NAME'] + """ (\
_FILENAME VARCHAR(16777216),\
_FILEDATE TIMESTAMP_NTZ(9),\
_UPLOADDATE TIMESTAMP_NTZ(9),\
JSONFIELD VARIANT\
)COMMENT='Model GMX """ + df_tables['TABLE_NAME'] + """ file.';"""

#print(df_tables['ddl'].values)

## output the LZ tables to file
outstr = ""
for idx, row in df_tables.iterrows():
    outstr += row['ddl'] + """
"""
print(outstr)
writeFile(path,outstr)

In [26]:
## create the gmx file fileupoader.config entries
#read existing file
configFileList = getFile(configFile)
#print(configFileStr)

# create new list, remove empty lines
newFileList = []
for l in configFileList:
    if l != "":
        newFileList.append(l)

# remove last line
newFileList = newFileList[:-1]

#create the new config entries
confstr = ""

for l in newFileList:
    confstr += l

confstr = confstr.rstrip()
confstr = confstr.rstrip('\r\n')
confstr = confstr.rstrip()

for idx, row in df_tables.iterrows():
    confTableName = row['TABLE_NAME'] 
    confstr += """,
    "SF_""" + confTableName + """\": {
        "SNOWFLAKE_SCHEMA_NAME": \"""" + targetSchema + """\",
        "SNOWFLAKE_DATABASE_NAME": "<<LZ>>",
        "SNOWFLAKE_WAREHOUSE_NAME": "Import_File_WH",
        "SNOWFLAKE_ROLE_NAME": "<<SYSADMIN>>",
        "SNOWFLAKE_TABLE_NAME": \"""" + confTableName + """\",
        "SNOWFLAKE_FILE_FORMAT_NAME": "CONTROL.POWERSHELL_JSON",
        "SNOWFLAKE_STAGE_TYPE": "NAMED",
        "SNOWFLAKE_STAGE_NAME": "SAPH_STG",
        "FILE": "F:\\\\Shares\\\\Files\\\\GMX\\\\USC\\\\SF_""" + confTableName + """_*.json",
        "SPLIT_FILE": false
    }"""

# add final bracket
confstr += """
}"""

# write out new config file
writeFile(configFile,confstr)

In [8]:
## Helper functions and variables for DBT model and schema.yml file

db = "LZ_DEV"
lzSchema = "SAPH_USR2"
odsSchema = "SAPH"
repo_path = "C:\\Repo\\EnterpriseCloud\\"

#CREATE DBT MODEL
#(inputs are column list for source_cte,transform_cte and tablename )
def getDbtCode(col_s_string,col_t_string,col_type_data_str,tablename,odsSchema,db):

	DBT_CODE = """

	/************
	Model file generated from gmxUtility 
	*************/

	{{ config(
	  materialized='incremental',
	  incremental_strategy='merge',
	  alias='""" + tablename + """',
	  unique_key = [],
	  transient=false
	) }}

	with source_data as (
	SELECT
		""" +col_s_string+ """,
		_FILEDATE,
		_FILENAME
	from """ + "{{ source('"+ odsSchema + "','" + tablename + "') }}" """
	{% if is_incremental() %}WHERE  _FileDate > IFNULL((SELECT MAX(_FileDate) FROM {{ this }}),'1970-01-01'){% endif %}

	),
	
	transform_data as (
	SELECT 
		""" +col_t_string+ """,
		_FILEDATE,
		_FILENAME,
		'{{ invocation_id }}' AS AUDIT_LOG_ID,
        SHA2('hash_me') as _ROW_HASH -- Adds column to table on full refresh.
	from source_data
	),
	
	type_data as (
	SELECT
		"""+col_type_data_str+""",
		cast(_FILEDATE AS TIMESTAMP_NTZ(9)) AS _FILEDATE,
		cast(_FILENAME as varchar(128)) as _FILENAME,
        cast(AUDIT_LOG_ID as varchar(128)) as AUDIT_LOG_ID,
        cast(_ROW_HASH as varchar(128)) as _ROW_HASH
	from transform_data
	)

	select * from type_data

			"""
	#print(DBT_CODE)
	return DBT_CODE

# #CREATE FILE IN THE REQUIRED LOCATION
# def createFile(path,filename):
#     if os.path.isdir(path):
#         with open(os.path.join(path, filename), 'w') as fp:
#             pass

#CREATE SCHEMA FILE
def createSchema(df_tables_test,lzSchema,odsSchema):
        df_tables_test['schema'] = """\n      - name: """ + df_tables_test['TABLE_NAME'] + """\n        identifier: """ + df_tables_test['TABLE_NAME'] + """\n        description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str = """version: 2\nsources:\n  - name: """ + odsSchema + """\n    description: Landing Zone - """ + lzSchema + """ Schema\n    database: "{{ env_var('SNOWFLAKE_LZ') }}"\n    schema: """+ lzSchema + """\n    tables:"""

        for idx, r in df_tables_test.iterrows():
            schema_str += r['schema'] + """
        """
        df_tables_test['models'] = """\n  - name: """ + df_tables_test['TABLE_NAME'] + """\n    identifier: """ + df_tables_test['TABLE_NAME'] + """\n    description:  Landing table for """ + df_tables_test['TABLE_NAME'] + """ file. """

        schema_str_t ="""\nmodels:"""
        for idx, r in df_tables_test.iterrows():
            schema_str_t += r['models'] + """
        """
        final_schema = schema_str + schema_str_t

        path_create = repo_path + "dbt\\lz_to_ods\\models\\saph"
        filename = "schema.yml"
        path_write = path_create +"\\"+ filename

        # createFile(path_create,filename)
        writeFile(path_write,final_schema)


In [9]:
#Getting load types from csv
# df_loadtype = pd.read_csv('C:\\Users\\221291\\Documents\\220629_SnowFlakeViewLoadDefinition.csv')
# df_loadtype = df_loadtype[["TableOrViewName","ObjectType"]]
# df_loadtype["TableOrViewName"] = df_loadtype["TableOrViewName"].str.upper()
csv_path = repo_path + "Notebooks\\Sf_keywords.csv"
df_keywords = pd.read_csv(csv_path)
print(df_keywords['KEYWORDS'])
df_keywords["KEYWORDS"] = df_keywords["KEYWORDS"].str.upper()


0      ACCOUNT
1          ALL
2        ALTER
3          AND
4          ANY
        ...   
86        VIEW
87        WHEN
88    WHENEVER
89       WHERE
90        WITH
Name: KEYWORDS, Length: 91, dtype: object


In [56]:
#iterate thru list of tables and create dbt model files

df_tables_test = pd.DataFrame(df_tables["TABLE_NAME"])


for tables_gmx in range(len(df_tables_test)):
   

    #sql_keys =""" select object_keys(value) from SAPH_USC.SAP_ZZM_EVF_ADDRESS_GMX limit 1;"""

    #query to fetch JSON KEYS (Column Names)
    sql_keys = """select f.key from table(flatten(input => parse_json(select JSONFIELD from """ + lzSchema + """.""" + df_tables_test.iloc[tables_gmx,0] + """ limit 1))) f;"""

    #snowflake call
    df_keys = df(conn,sql_keys)

    #skipping empty tables
    if df_keys.empty:
        continue
    
    

    #empty list for transform_cte,source_cte,type_data_cte columns 
    col_t = []
    col_s = []
    col_type_data = []

    df_keys_nokeyword = df_keys.copy()

    
    #add underscore to keywords
    for i in range(len(df_keys_nokeyword)):
        
        #replace characters with underscores
        df_keys_nokeyword.iloc[i,0] = df_keys_nokeyword.iloc[i,0].replace('-','_')

        #modify keywords
        for k in range(len(df_keywords)):
            if df_keys_nokeyword.iloc[i,0].upper() == df_keywords.iloc[k,0]:
                #print(df_keys_keyword.iloc[i,0])
                
                df_keys_nokeyword.iloc[i,0] = "_" + df_keys_nokeyword.iloc[i,0]

                break

    #type data columns
    col_type_data = df_keys_nokeyword["KEY"].to_list()
    for i in range(len(col_type_data)):
        col_type_data[i] = 'CAST("' + col_type_data[i] + '" AS VARCHAR(16777216)) AS "' + col_type_data[i] + '"'
    col_type_data_str = ', \n \t \t'.join(col_type_data)
    col_type_data_str = col_type_data_str.upper()
    
    # column string for transform cte
    col_t=df_keys_nokeyword["KEY"].to_list()
    col_t_string = '"' + '", \n \t \t"'.join(col_t) + '"'
    col_t_string = col_t_string.upper()
    # column string for source cte 
    
            

    for i in range(len(df_keys)):
        df_keys.iloc[i, 0] = 'JSONFIELD:"' + df_keys.iloc[i, 0] + '"::STRING AS "' + df_keys_nokeyword.iloc[i, 0].upper() + '"'
    col_s = []
    col_s = df_keys["KEY"].to_list()
    col_s_string = ', \n \t \t'.join(col_s)

    #Generate load type (kept for future)
    # for j in range(len(df_loadtype)):
    #     if df_tables_test.iloc[tables_gmx,0] == df_loadtype.iloc[j,0]:
    #         load_type = df_loadtype.iloc[j,1]
    #         break

    #Creating file
    dbt_model = getDbtCode(col_s_string,col_t_string,col_type_data_str,df_tables_test.iloc[tables_gmx,0],odsSchema,db)
    
    
    dbt_path = repo_path + "dbt\\lz_to_ods\\models\\saph\\"
    filename = df_tables_test.iloc[tables_gmx,0] + ".sql"
    path_write = dbt_path + filename

    writeFile(path_write,dbt_model)


In [62]:
#Create Schema.yml file
lzSchema = "SAPH_USR2"
odsSchema = "SAPH"
createSchema(df_tables_test,lzSchema,odsSchema)

In [ ]:
## create pub ODS_ views
%run ..\\Python\\dbtTools\\dbtGeneratePubViews.py "ODS_DBT" "SAPH" "C:\Source\Repos\EnterpriseCloud\dbt\ods_to_pub\models" "eric.x.coleman@us.tel.com"


In [10]:
#fetch loadtype
repo_path = "C:\\Repo\\EnterpriseCloud\\"

loadtype_csv_path = repo_path + "Notebooks\\220629_SnowFlakeViewLoadDefinition.csv"
df_loadtype = pd.read_csv(loadtype_csv_path)
df_loadtype = df_loadtype[["TableOrViewName","ObjectType"]]
df_loadtype["TableOrViewName"] = df_loadtype["TableOrViewName"].str.upper()

for i in range(len(df_loadtype)):
        df_loadtype.iloc[i,1] = df_loadtype.iloc[i,1].replace('Truncate  insert','insert_overwrite')
        df_loadtype.iloc[i,1] = df_loadtype.iloc[i,1].replace('Incremental','merge')
        df_loadtype.iloc[i,0] = df_loadtype.iloc[i,0].replace('SF_','')
print(df_loadtype)

                TableOrViewName        ObjectType
0          SAP_US_FIT_EV_ACDOCA             merge
1      SAP_US_MMT_EV_MATDOC_GMX             merge
2    PARTS_EXPORTIMPORT_CONTROL  insert_overwrite
3    PSL_ZZM_EV_FISCALPERIODEXT  insert_overwrite
4   SAP_AAM_EVF_ASSETNUMBER_GMX  insert_overwrite
..                          ...               ...
89          SAP_SDM_EV_A904_GMX  insert_overwrite
90          SAP_SDM_EV_A902_GMX  insert_overwrite
91          SAP_SDM_EV_A901_GMX  insert_overwrite
92              SAP_COM_EV_CEPC  insert_overwrite
93              SAP_FIM_EV_SKB1  insert_overwrite

[94 rows x 2 columns]


In [14]:
dbt_path = repo_path + "dbt\\lz_to_ods\\models\\saph\\"
filesList= getFileList(dbt_path)

for file in filesList:
    filestr = ''
    filein = getFile(file)

    for line in filein:        
        if "alias" in line or 'alias =' in line:
            tableName = line.replace("alias=","")
            tableName = tableName.replace("'","")
            tableName = tableName.replace(",","")
            tableName = tableName.strip()
            
        if "incremental_strategy=" in line or 'incremental_strategy =' in line:
            load_type = line.replace("incremental_strategy=","")
            load_type = load_type.replace("'","")
            load_type = load_type.replace(",","")
            load_type = load_type.strip()   

    # for j in range(len(df_loadtype)):
    #     if tableName == df_loadtype.iloc[j,0]:
    #         load_type = df_loadtype.iloc[j,1]
    #         break
        

    for line in filein:
        filestr += line
    
    if load_type == 'insert_overwrite':
        #load_type = "incremental_strategy=" +"'" + load_type + "'"
        #filestr = filestr.replace("incremental_strategy='merge'",load_type)
        #replace_str = """unique_key = [],
	  #transient=false"""
    #     replace_str = """from {{ source('SAPH',"""+"'"+tableName+"'"+""") }} s
	# join max_d on s._filedate = max_d._filedate
	#  and s._filename = max_d._filename"""
    #     replacement_str = """from {{ source('SAPH', """+"'"+tableName+"'"+""") }}
	# WHERE _FILENAME = (SELECT MAX(_FILENAME) from {{ source('SAPH', """+"'"+tableName+"'"+""") }})"""
        replace_str_a = """with max_f as (
		SELECT MAX(_FILENAME) as _filename 
		from {{ source("SAPH", """'"'+tableName+'"'""") }}
	),



    

	max_d as (
		select s._filename, max(s._filedate) as _filedate from {{ source("SAPH", """'"'+tableName+'"'""") }} as s
		join max_f on s._filename = max_f._filename
		group by s._filename
	),
	
	source_data"""
        replacement_str_a = """with source_data"""
        
        replace_str = """
        s._FILEDATE,
		s._FILENAME
    from {{ source('SAPH',"""+"'"+tableName+"'"+""") }} s
	join max_d on s._filedate = max_d._filedate 
     and s._filename = max_d._filename"""
        
        replacement_str = """_FILEDATE,
		_FILENAME
	from {{ source('SAPH',"""+"'"+tableName+"'"+""") }}
	WHERE REGEXP_SUBSTR(_FILENAME,'\\d{14}') = (SELECT MAX(REGEXP_SUBSTR(_filename,'\\d{14}')) from {{ source('SAPH', """+"'"+tableName+"'"+""") }})"""

        
        filestr = filestr.replace(replace_str_a,replacement_str_a).replace(replace_str,replacement_str)
   

    
        
        writeFile(file,filestr)
    else:
        pass


In [13]:
repo_path = "C:\\Repo\\EnterpriseCloud\\"

dbt_path = repo_path + "dbt\\lz_to_ods\\models\\saph\\"
filesList= getFileList(dbt_path)

for file in filesList:
    filestr = ''
    filein = getFile(file)

    for line in filein:        
        # if "alias" in line or 'alias =' in line:
        #     tableName = line.replace("alias=","")
        #     tableName = tableName.replace("'","")
        #     tableName = tableName.replace(",","")
        #     tableName = tableName.strip()
            
        # if "incremental_strategy=" in line or 'incremental_strategy =' in line:
        #     load_type = line.replace("incremental_strategy=","")
        #     load_type = load_type.replace("'","")
        #     load_type = load_type.replace(",","")
        #     load_type = load_type.strip()
        if "with max_f as (" in line:
            print(file)
    # for j in range(len(df_loadtype)):
    #     if tableName == df_loadtype.iloc[j,0]:
    #         load_type = df_loadtype.iloc[j,1]
    #         break
        

    for line in filein:
        filestr += line
    
    # if load_type == 'insert_overwrite':
    #     print(tableName + load_type)

C:\Repo\EnterpriseCloud\dbt\lz_to_ods\models\saph\\PARTS_EXPORTIMPORT_CONTROL.sql
C:\Repo\EnterpriseCloud\dbt\lz_to_ods\models\saph\\SAP_COM_EV_PRTE.sql
C:\Repo\EnterpriseCloud\dbt\lz_to_ods\models\saph\\SAP_SDM_EV_A904_GMX.sql
C:\Repo\EnterpriseCloud\dbt\lz_to_ods\models\saph\\SAP_SDM_EV_KNVP.sql
C:\Repo\EnterpriseCloud\dbt\lz_to_ods\models\saph\\SAP_ZZM_EV_BUT000.sql
C:\Repo\EnterpriseCloud\dbt\lz_to_ods\models\saph\\SAP_ZZM_EV_INOB.sql


In [15]:
repo_path = "C:\\Repo\\EnterpriseCloud\\"
dbt_path = repo_path + "dbt\\lz_to_ods\\models\\sapapi\\"
filesList= getFileList(dbt_path)

for file in filesList:
    filestr = ''
    filein = getFile(file)
    replaced_content = ""
    new_line = ""

    for line in filein: 
        line = line.strip()
       
        if "{% if is_incremental() %}WHERE  _FILENAME > IFNULL((SELECT MAX(_FILENAME) FROM {{ this }}),'1970-01-01'){% endif %}" in line:
            
            #replacing the texts
            new_line = line.replace("{% if is_incremental() %}WHERE  _FILENAME > IFNULL((SELECT MAX(_FILENAME) FROM {{ this }}),'1970-01-01'){% endif %}", "{% if is_incremental() %}WHERE  _FILEDATE > IFNULL((SELECT MAX(_FILEDATE) FROM {{ this }}),'1970-01-01'){% endif %}")
        else:
            new_line = line    #concatenate the new string and add an end-line break
        replaced_content = replaced_content + new_line + "\n"
        
    writeFile(file,replaced_content)